In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from rich import print
from loguru import logger

class OverAllState(TypedDict):
    final_res: str

def node_a(state: OverAllState) -> OverAllState:
    logger.info("node_a が実行されました")
    return {
        "final_res": "node_a 実行の中間結果"
    }

def node_b(state: OverAllState) -> OverAllState:
    logger.info("node_b が実行されました")
    return {
        "final_res": "node_b 実行の中間結果"
    }

def node_c(state: OverAllState) -> OverAllState:
    logger.info("node_c が実行されました")
    return {
        "final_res": "node_c 実行の中間結果"
    }


builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_node("node_c", node_c)

builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", "node_c")
builder.add_edge("node_c", END)

checkpointer = InMemorySaver()
graph = builder.compile(
    checkpointer=checkpointer
)

from IPython.display import display
display(graph)

config = {"configurable": {"thread_id": "123"}}

logger.info("{}-> 1回目の実行 <-{}", "=" * 10, "=" * 10)
first_res = graph.invoke(
    {},
    config=config,
    interrupt_before=["node_a", "node_b"],
    interrupt_after=["node_a", "node_b"]
)
logger.info("1回目の実行結果：{}", first_res)

logger.info("{}-> 2回目の実行 <-{}", "=" * 10, "=" * 10)
second_res = graph.invoke(
    None,
    config=config,
    interrupt_before=["node_a", "node_b"],
    interrupt_after=["node_a", "node_b"]
)
logger.info("2回目の実行結果：{}", second_res)

logger.info("{}-> 3回目の実行 <-{}", "=" * 10, "=" * 10)
third_res = graph.invoke(None, config=config)
logger.info("3回目の実行結果：{}", third_res)

logger.info("{}-> 4回目の実行 <-{}", "=" * 10, "=" * 10)
final_res = graph.invoke(None, config=config)
logger.info("4回目の実行結果：{}", final_res)

logger.info("{}-> 完了 <-{}", "=" * 10, "=" * 10)